In [ ]:
port torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from google.colab.patches import cv2_imshow
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
!wget https://www.agentspace.org/download/fgsegmentation.pth

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git

In [ ]:
sys.path.append("dinov3")

In [ ]:
# expander

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(), #nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(), #nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, scale):
        super().__init__()
        self.scale = scale
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False)
        x = self.conv(x)
        return x


class FeatureExpander(nn.Module):
    def __init__(self):
        super().__init__()

        self.up1 = UpsampleBlock(384, 256, scale=2)   # 128 → 256
        self.up2 = UpsampleBlock(256, 128, scale=3)   # 256 → 768
        self.out = nn.Conv2d(128, 32, kernel_size=1) #16

    def forward(self, x):
        # input: B x 384 x 128 x 128
        x = self.up1(x)   # B x 256 x 256 x 256
        x = self.up2(x)   # B x 128 x 768 x 768
        x = self.out(x)   # B x 16 x 768 x 768
        return x

In [ ]:
!pip install fastncut

In [ ]:
from fastncut import Ncut

In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, backbone, expander):
        super().__init__()
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]
        self.backbone = backbone
        self.size = (768, 768)
        self.patch_size = backbone.patch_size
        self.expander = expander
        self.ncut = Ncut(num_iters=1)

    def preprocess(self, image):
        return torch.tensor(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).float().permute(2,0,1).unsqueeze(0)

    def postprocess(self, bipartition):
        return bipartition.cpu().numpy().astype(np.uint8)*255

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.float()/255
        x = F.interpolate(x, size=self.size, mode='bilinear', align_corners=False)
        x[:,0] = (x[:,0] - self.mean[0]) / self.std[0]
        x[:,1] = (x[:,1] - self.mean[1]) / self.std[1]
        x[:,2] = (x[:,2] - self.mean[2]) / self.std[2]
        x = self.backbone.get_intermediate_layers(x,norm=False)[0]
        x = x.reshape(B,self.size[0]//self.patch_size,self.size[1]//self.patch_size,-1).permute(0,3,1,2)
        x = self.expander(x)
        x = F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)
        x = self.ncut(x)
        return x

In [ ]:
model = torch.load('fgsegmentation.pth',weights_only=False).to(device)
model.eval()

In [ ]:
!wget -O input.jpg https://github.com/andylucny/BackgroundRemovalByModNet4OpenCV/blob/main/input.jpg?raw=true

In [ ]:
input = cv2.imread('input.jpg')

In [ ]:
blob = model.preprocess(input).to(device)

In [ ]:
output = model(blob)

In [ ]:
mask = model.postprocess(output[0])

In [ ]:
def cv2_imshow_mask(image, mask, display=True):
    overlay = image.copy()
    overlay[mask > 0] = (0,255,255)
    result = cv2.addWeighted(overlay, 0.4, image, 0.6, 0)
    if display:
        cv2_imshow(result)
    else:
        return result

In [ ]:
plt.imshow(cv2.cvtColor(cv2_imshow_mask(input, mask, display=False),cv2.COLOR_BGR2RGB))

In [ ]:
# download dataset https://github.com/JizhiziLi/AIM
# Li, Jizhizi, Jing Zhang, and Dacheng Tao. "Deep Automatic Natural Image Matting." IJCAI. 2021.
# refer to the paper (https://arxiv.org/abs/2107.07235), the GitHub repo (https://github.com/JizhiziLi/AIM) or contacts Jizhizi Li at jili8515@uni.sydney.edu.au
!wget http://www.agentspace.org/download/AIM-500-20260316T160703Z-1-001.zip
!unzip AIM-500-20260316T160703Z-1-001.zip > /dev/null

In [ ]:
def IoU(target,output):
    target = target.astype(bool)
    output = output.astype(bool)
    intersection = np.logical_and(target, output)
    union = np.logical_or(target, output)
    iou_score = np.sum(intersection) / np.sum(union)
    return iou_score

In [ ]:
def MSE(target,output):
    target = target.astype(float)/255
    output = output.astype(float)/255
    return np.mean((target-output)**2)

In [ ]:
# find all images
ious = []
mses = []
for root, dirs, files in os.walk("AIM-500/original"):
    for file in files:
        if file.endswith(".jpg"):
            image = cv2.imread(os.path.join("AIM-500/original",file))
            target_mask = cv2.imread(os.path.join("AIM-500/mask",file[:-4]+'.png'),cv2.IMREAD_GRAYSCALE)
            threshold, target_mask = cv2.threshold(target_mask,0,255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            blob = model.preprocess(image).to(device)
            output = model(blob)
            mask = model.postprocess(output[0])
            iou = IoU(target_mask,mask)
            ious.append(iou)
            mse = MSE(target_mask,mask)
            mses.append(mse)
            if len(ious) % 100 == 0:
                print(len(ious),iou,mse)
print(np.mean(ious))
print(np.mean(mses))